# ChatUniTest LoRA Fine-tuning

**目标**：基于 CodeLlama-7b-Instruct + QLoRA 微调，生成 Java JUnit 测试

**预计时间**：1.5–2 小时（A100）

**预计成本**：~$2–3

**运行前检查**：
- Runtime → Change runtime type → **A100 GPU**
- 确保 HuggingFace token 已准备好（需要 write 权限）

## Step 1：安装依赖

In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 trl==0.8.6 \
    bitsandbytes==0.43.1 accelerate==0.29.3 \
    datasets sentencepiece huggingface_hub

# 验证 GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2：登录 HuggingFace（需要 write token）

In [ ]:
from huggingface_hub import login, notebook_login

# 方式一：交互式登录（推荐）
notebook_login()

# 方式二：直接填入 token（不推荐明文存储）
# login(token="hf_xxxxxxxxxxxxx")

## Step 3：准备数据集

In [ ]:
from datasets import load_dataset
import pandas as pd
import re

# ── Prompt 模板（与 model_server.py 推理格式一致）──
PROMPT_TEMPLATE = """mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
{context}
runtimeFacts:

### JUnit Test:
"""

def build_full_text(context: str, test: str) -> str:
    return PROMPT_TEMPLATE.format(context=context.strip()) + test.strip()

def has_assertion(test: str) -> bool:
    return any(kw in test for kw in ["assert", "Assert", "verify", "Verify", "fail("])

def is_valid_java(code: str) -> bool:
    return code.count("{") > 0 and abs(code.count("{") - code.count("}")) <= 2

def estimate_tokens(text: str) -> int:
    return len(text) // 4

# 加载数据集
print("Loading dataset...")
raw = load_dataset("zzzghttt/context2test", split="train")
print(f"Raw samples: {len(raw)}")
print(f"Columns: {raw.column_names}")

# 自动检测字段名
col_context = next((c for c in ["context", "input", "source"] if c in raw.column_names), None)
col_test    = next((t for t in ["test", "output", "target"] if t in raw.column_names), None)
print(f"Using: context='{col_context}', test='{col_test}')"

df = raw.to_pandas()[[col_context, col_test]].copy()
df.columns = ["context", "test"]

# ── 数据清洗 ──
before = len(df)
df = df.drop_duplicates(subset=["context"])
print(f"去重后: {len(df)} (移除 {before - len(df)})")

df = df[df["test"].apply(has_assertion)]
print(f"过滤无断言后: {len(df)}")

df = df[df["test"].apply(is_valid_java)]
print(f"过滤结构异常后: {len(df)}")

df = df[df["context"].str.strip().str.len() > 30]
print(f"过滤空 context 后: {len(df)}")

# 构建完整训练文本
df["text"] = df.apply(lambda r: build_full_text(r["context"], r["test"]), axis=1)
df["token_est"] = df["text"].apply(estimate_tokens)
df = df[df["token_est"] <= 2048]
print(f"过滤超长后: {len(df)}")

# 随机取 5000 条
df = df.sample(frac=1, random_state=42).reset_index(drop=True).head(5000)
print(f"\n最终训练样本数: {len(df)}")

# 转为 HuggingFace Dataset
from datasets import Dataset
train_dataset = Dataset.from_pandas(df[["text"]])

# 预览第一条
print("\n=== 样本预览（前 500 字符）===")
print(train_dataset[0]["text"][:500])

## Step 4：加载 Base Model（QLoRA 4-bit）

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

BASE_MODEL = "codellama/CodeLlama-7b-Instruct-hf"

# 4-bit QLoRA 配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded. 显存占用: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 5：配置 LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 为 QLoRA 训练准备模型
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,                    # 原始 TestGen2-lora 用 64，这里用 32 减少训练时间
    lora_alpha=64,           # alpha = 2*r，提升稳定性
    target_modules=[
        "q_proj", "v_proj",  # 原始配置
        "k_proj", "o_proj",  # 扩展：覆盖完整 attention
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 预期输出：trainable params: ~8M / 7B total (~0.1%)

## Step 6：训练

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# ── 修改这里：你的 HuggingFace 用户名 ──
HF_USERNAME = "your-hf-username"   # <-- 改成你的
OUTPUT_MODEL = f"{HF_USERNAME}/my-testgen-lora"

training_args = TrainingArguments(
    output_dir="./my-testgen-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,       # 等效 batch_size = 32
    gradient_checkpointing=True,         # 节省显存
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=False,
    bf16=True,                           # A100 支持 bfloat16
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,                      # 防断线，每 100 步保存 checkpoint
    save_total_limit=3,
    report_to="none",                    # 不上传 wandb
    optim="paged_adamw_32bit",           # QLoRA 推荐优化器
    max_grad_norm=0.3,
    group_by_length=True,                # 相近长度的样本分组，减少 padding
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048,                 # 与推理 cutoff_len 一致
    packing=False,
)

print("开始训练...")
print(f"总步数: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
trainer.train()

## Step 7：保存并推送到 HuggingFace Hub

In [ ]:
print("保存模型...")
trainer.save_model("./my-testgen-lora")

print(f"推送到 HuggingFace Hub: {OUTPUT_MODEL}")
model.push_to_hub(OUTPUT_MODEL, private=False)
tokenizer.push_to_hub(OUTPUT_MODEL, private=False)

print(f"\n完成！模型地址：https://huggingface.co/{OUTPUT_MODEL}")
print(f"\n下一步：在 model_server.py 第 23 行替换：")
print(f'  PeftModel.from_pretrained(model, "{OUTPUT_MODEL}")')

## Step 8（可选）：从断点恢复训练

如果 Colab 断线，用以下代码从最近的 checkpoint 恢复：

In [ ]:
import os

# 找到最新的 checkpoint
checkpoints = [
    d for d in os.listdir("./my-testgen-lora")
    if d.startswith("checkpoint-")
]
if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    resume_from = f"./my-testgen-lora/{latest}"
    print(f"从 {resume_from} 恢复训练")
    trainer.train(resume_from_checkpoint=resume_from)
else:
    print("没有找到 checkpoint，请从头开始训练")

## Step 9（可选）：快速验证生成效果

In [ ]:
from transformers import GenerationConfig

# 用与 model_server.py 相同的 prompt 格式测试
test_prompt = """mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
public class PDFTextStripper {
    public String getText(PDDocument doc) throws IOException {
        StringWriter writer = new StringWriter();
        writeText(doc, writer);
        return writer.toString();
    }
}
runtimeFacts:

### JUnit Test:
"""

model.eval()
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.6,
        do_sample=True,
        top_p=0.95,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== 生成结果 ===")
print(generated)